## HC3 Train / Validation / Test Split

### Load cleaned dataset

In [46]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_dataset.csv")

### Analyze the cleaned dataset output

In [47]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
df.head()

Shape: (23867, 5)

Columns:
['id', 'question', 'human_answers', 'chatgpt_answers', 'source']


,id,question,human_answers,chatgpt_answers,source
0,0,"Why is every book I hear about a "" NY Times # ...","['Basically there are many categories of "" Bes...",['There are many different best seller lists t...,reddit_eli5
1,1,"If salt is so bad for cars , why do we use it ...",['salt is good for not dying in car crashes an...,"[""Salt is used on roads to help melt ice and s...",reddit_eli5
2,2,Why do we still have SD TV channels when HD lo...,"[""The way it works is that old TV stations got...","[""There are a few reasons why we still have SD...",reddit_eli5
3,3,Why has nobody assassinated Kim Jong - un He i...,"[""You ca n't just go around assassinating the ...",['It is generally not acceptable or ethical to...,reddit_eli5
4,4,How was airplane technology able to advance so...,['Wanting to kill the shit out of Germans driv...,['After the Wright Brothers made the first pow...,reddit_eli5


In [48]:
print("Number of rows:", len(df))
print("Number of unique IDs:", df["id"].nunique())
print("Duplicate IDs:", df["id"].duplicated().sum())

print("\nHuman answers data type:")
print(type(df.loc[0, "human_answers"]))

print("\nExample human answers:")
print(df.loc[0, "human_answers"])

print("\nExample ChatGPT answers:")
print(df.loc[0, "chatgpt_answers"])

Number of rows: 23867
Number of unique IDs: 23867
Duplicate IDs: 0

Human answers data type:
<class 'str'>

Example human answers:
['Basically there are many categories of " Best Seller " . Replace " Best Seller " by something like " Oscars " and every " best seller " book is basically an " oscar - winning " book . May not have won the " Best film " , but even if you won the best director or best script , you \'re still an " oscar - winning " film . Same thing for best sellers . Also , IIRC the rankings change every week or something like that . Some you might not be best seller one week , but you may be the next week . I guess even if you do n\'t stay there for long , you still achieved the status . Hence , # 1 best seller .' "If you 're hearing about it , it 's because it was a very good or very well - publicized book ( or both ) , and almost every good or well - publicized book will be # 1 on the NY Times bestseller list for at least a little bit . Kindof like how almost every big o

## Train, Validation, and Test Split

The dataset is split into training, validation, and test sets using an
80/10/10 ratio.

Each row represents one unique question and contains both human and ChatGPT
answers. Splitting at the question level keeps all answers associated with
the same question in the same dataset and prevents question-level data
leakage.

The split is stratified by source so that the five HC3 domains maintain
approximately the same distribution across the training, validation, and
test sets.

A fixed random state of 42 is used to make the split reproducible.

In [49]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

In [50]:
train, temp = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["source"]
)

validation, test = train_test_split(
    temp,
    test_size=0.5,
    random_state=42,
    stratify=temp["source"]
)

### Check the split sizes

In [51]:
print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

print("\nTotal:", len(train) + len(validation) + len(test))

Train: 19093
Validation: 2387
Test: 2387

Total: 23867


### Verify the stratification

In [52]:
print("Train source distribution:")
print(train["source"].value_counts(normalize=True).round(3))

print("\nValidation source distribution:")
print(validation["source"].value_counts(normalize=True).round(3))

print("\nTest source distribution:")
print(test["source"].value_counts(normalize=True).round(3))

Train source distribution:
source
reddit_eli5    0.698
finance        0.165
medicine       0.052
open_qa        0.050
wiki_csai      0.035
Name: proportion, dtype: float64

Validation source distribution:
source
reddit_eli5    0.698
finance        0.165
medicine       0.052
open_qa        0.050
wiki_csai      0.035
Name: proportion, dtype: float64

Test source distribution:
source
reddit_eli5    0.698
finance        0.165
medicine       0.052
open_qa        0.049
wiki_csai      0.035
Name: proportion, dtype: float64


### Check for question leakage

In [53]:
train_ids = set(train["id"])
validation_ids = set(validation["id"])
test_ids = set(test["id"])

print("Train / Validation overlap:",
      len(train_ids.intersection(validation_ids)))

print("Train / Test overlap:",
      len(train_ids.intersection(test_ids)))

print("Validation / Test overlap:",
      len(validation_ids.intersection(test_ids)))

Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0


### Final summary

In [55]:
print("Final dataset split")
print("-------------------")
print(f"Training:   {len(train)} questions")
print(f"Validation: {len(validation)} questions")
print(f"Test:       {len(test)} questions")
print(f"Total:      {len(train) + len(validation) + len(test)} questions")

Final dataset split
-------------------
Training:   19093 questions
Validation: 2387 questions
Test:       2387 questions
Total:      23867 questions


### Saving the splits

In [56]:
train.to_csv("../data/processed/train.csv", index=False)
validation.to_csv("../data/processed/validation.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)

print("Dataset splits saved successfully.")

Dataset splits saved successfully.


## Split Summary

The cleaned HC3 dataset was divided into training, validation, and test
sets using an approximately 80/10/10 split.

- Splitting was performed at the question level.
- Source/domain distributions were preserved using stratified sampling.
- A random state of 42 makes the split reproducible.
- No question IDs overlap between the training, validation, and test sets.
- Each question and all of its associated human and ChatGPT answers remain
  together in a single split.

These splits can now be used for downstream answer expansion, labeling,
TF-IDF feature engineering, and model development.

In [58]:
print("Empty ChatGPT answers by split:")
print("Train:", (train["chatgpt_answers"] == "[]").sum())
print("Validation:", (validation["chatgpt_answers"] == "[]").sum())
print("Test:", (test["chatgpt_answers"] == "[]").sum())

Empty ChatGPT answers by split:
Train: 0
Validation: 0
Test: 0


In [59]:
print("Empty human answer lists:")
print((df["human_answers"] == "[]").sum())

print("\nEmpty ChatGPT answer lists:")
print((df["chatgpt_answers"] == "[]").sum())

Empty human answer lists:
0

Empty ChatGPT answer lists:
0
